In [42]:
import pandas as pd
import numpy as np
import sys

sys.path.append("../src")

import clean

data_folder = "../data/cleaned/"

names = ["orders", "order_items", "customers", "restaurants", "menu",
         "delivery_partners", "payments", "customer_feedback",
         "promotions", "cities", "traffic", "weather"]

tables = {}
clean_tables = {}

for n in names:
    tables[n] = pd.read_csv(data_folder + n + ".csv")
    clean_tables[n] = getattr(clean, "clean_" + n)(tables[n])

abt = clean.build_abt(clean_tables)

print("ABT shape:", abt.shape)

ABT shape: (20475, 57)


In [43]:
import features
import importlib
importlib.reload(features)

abt = features.add_time_features(abt)

print("ABT shape:", abt.shape)
print()
print("PeakHour:")
print(abt["PeakHour"].value_counts().sort_index())
print()
print("WeekendOrder:")
print(abt["WeekendOrder"].value_counts().sort_index())
print()
print("MealTime:")
print(abt["MealTime"].value_counts())
print()
print(abt[["OrderDate", "OrderTime", "OrderHour", "PeakHour", "DayOfWeek", "WeekendOrder", "OrderMonth", "MealTime"]].head(5))

ABT shape: (20475, 63)

PeakHour:
PeakHour
0    10728
1     9747
Name: count, dtype: int64

WeekendOrder:
WeekendOrder
0    14684
1     5791
Name: count, dtype: int64

MealTime:
MealTime
Dinner        6016
Lunch         5940
Breakfast     3622
Snack         2882
Late Night    2015
Name: count, dtype: int64

   OrderDate OrderTime  OrderHour  PeakHour DayOfWeek  WeekendOrder  \
0 2023-06-17  11:51:00         11         1  Saturday             1   
1 2023-07-01  10:18:00         10         0  Saturday             1   
2 2023-05-09  19:34:00         19         1   Tuesday             0   
3 2024-06-22  00:26:00          0         0  Saturday             1   
4 2024-03-17  12:27:00         12         1    Sunday             1   

   OrderMonth    MealTime  
0           6       Lunch  
1           7   Breakfast  
2           5      Dinner  
3           6  Late Night  
4           3       Lunch  


In [44]:
import importlib
importlib.reload(features)

abt = features.add_tenure_features(abt)

print("ABT shape:", abt.shape)
print()
print("CustomerTenure:")
print(abt["CustomerTenure"].describe().round(1))
print("Missing:", abt["CustomerTenure"].isnull().sum())
print()
print("PartnerExperienceDays:")
print(abt["PartnerExperienceDays"].describe().round(1))
print("Missing:", abt["PartnerExperienceDays"].isnull().sum())

ABT shape: (20475, 65)

CustomerTenure:
count    13726.0
mean       394.0
std        259.6
min          0.0
25%        176.0
50%        364.0
75%        581.0
max       1088.0
Name: CustomerTenure, dtype: float64
Missing: 6749

PartnerExperienceDays:
count    13745.0
mean       485.3
std        311.4
min          0.0
25%        225.0
50%        455.0
75%        709.0
max       1401.0
Name: PartnerExperienceDays, dtype: float64
Missing: 6730


In [45]:
import importlib
importlib.reload(features)

abt = features.add_order_features(abt)

print("ABT shape:", abt.shape)
print()
print("HasCoupon:")
print(abt["HasCoupon"].value_counts().sort_index())
print()
print("DiscountPercent by coupon use:")
print(abt.groupby("HasCoupon")["DiscountPercent"].describe().round(1))
print()
print("DeliveryEfficiency:")
print(abt["DeliveryEfficiency"].describe().round(2))
print("Missing:", abt["DeliveryEfficiency"].isnull().sum())

ABT shape: (20475, 68)

HasCoupon:
HasCoupon
0    13268
1     7207
Name: count, dtype: int64

DiscountPercent by coupon use:
             count  mean   std   min   25%   50%   75%   max
HasCoupon                                                   
0          13268.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
1           7207.0  27.7  12.8  10.0  15.0  25.0  40.0  50.0

DeliveryEfficiency:
count    20332.00
mean         1.02
std          0.57
min          0.10
25%          0.66
50%          0.91
75%          1.23
max          8.02
Name: DeliveryEfficiency, dtype: float64
Missing: 143


In [46]:
import importlib
importlib.reload(features)

abt_final = features.build_features(clean.build_abt(clean_tables))

print("ABT shape:", abt_final.shape)
print("Same columns as the step-by-step version:", list(abt_final.columns) == list(abt.columns))
print("Duplicated OrderID:", abt_final["OrderID"].duplicated().sum())
print()

new_cols = ["OrderHour", "PeakHour", "DayOfWeek", "WeekendOrder", "OrderMonth", "MealTime",
            "CustomerTenure", "PartnerExperienceDays", "HasCoupon", "DiscountPercent", "DeliveryEfficiency"]

summary = pd.DataFrame({
    "type": abt_final[new_cols].dtypes.astype(str),
    "missing": abt_final[new_cols].isnull().sum(),
    "unique": abt_final[new_cols].nunique()
})
print(summary)
print()

print("Average delivery time (minutes):")
print("Peak hour vs normal hour:")
print(abt_final.groupby("PeakHour")["DeliveryTimeMinutes"].mean().round(1))
print("Weekend vs weekday:")
print(abt_final.groupby("WeekendOrder")["DeliveryTimeMinutes"].mean().round(1))

ABT shape: (20475, 68)
Same columns as the step-by-step version: True
Duplicated OrderID: 0

                          type  missing  unique
OrderHour                int64        0      24
PeakHour                 int64        0       2
DayOfWeek                  str        0       7
WeekendOrder             int64        0       2
OrderMonth               int32        0      12
MealTime                   str        0       5
CustomerTenure         float64     6749    1061
PartnerExperienceDays  float64     6730    1271
HasCoupon                int64        0       2
DiscountPercent        float64        0       8
DeliveryEfficiency     float64      143     433

Average delivery time (minutes):
Peak hour vs normal hour:
PeakHour
0    36.7
1    36.8
Name: DeliveryTimeMinutes, dtype: float64
Weekend vs weekday:
WeekendOrder
0    36.7
1    36.9
Name: DeliveryTimeMinutes, dtype: float64


In [47]:
print("1. ROW COUNTS (original CSV | after cleaning)")
for n in names:
    print(n, "|", len(tables[n]), "|", len(clean_tables[n]))
print("ABT rows:", len(abt_final))
print()

o = tables["orders"]
oc = clean_tables["orders"]

print("2. MONEY TOTALS (orders table)")
print("SUM FinalAmount:", round(o["FinalAmount"].sum(), 2))
print("SUM FoodCost:", int(o["FoodCost"].sum()))
print("SUM DeliveryFee:", int(o["DeliveryFee"].sum()))
print("SUM Discount:", round(o["Discount"].sum(), 2))
print("SUM GST:", round(o["GST"].sum(), 2))
print("ABT SUM FinalAmount:", round(abt_final["FinalAmount"].sum(), 2))
print()

print("3. ORDERS PER STATUS")
print(o["OrderStatus"].value_counts().sort_index())
print()

print("4. DELIVERY TIME")
print("AVG DeliveryTimeMinutes, original:", round(o["DeliveryTimeMinutes"].mean(), 2))
print("AVG DeliveryTimeMinutes, after cleaning:", round(oc["DeliveryTimeMinutes"].mean(), 2))
print("Orders above 200 min in the original:", (o["DeliveryTimeMinutes"] > 200).sum())
print()

c = tables["customers"]
cc = clean_tables["customers"]

print("5. CUSTOMERS")
print("Customers:", len(c))
print("Pincode = 0, original:", (c["Pincode"] == 0).sum())
print("Pincode = 0, after cleaning:", (cc["Pincode"] == "0").sum())
print("Missing Pincode, original:", c["Pincode"].isnull().sum())
print("Missing Pincode, after cleaning:", cc["Pincode"].isnull().sum())
print("Missing Phone, original:", c["Phone"].isnull().sum())
print("Missing Phone, after cleaning:", cc["Phone"].isnull().sum())
print()

print("6. OTHER TOTALS")
print("SUM order_items TotalPrice:", round(tables["order_items"]["TotalPrice"].sum(), 2))
print("Distinct customers who ordered:", o["CustomerID"].nunique())
print("Distinct restaurants with orders:", o["RestaurantID"].nunique())
print("Orders with a coupon:", o["CouponCode"].notnull().sum())

1. ROW COUNTS (original CSV | after cleaning)
orders | 20475 | 20475
order_items | 42361 | 42361
customers | 12000 | 12000
restaurants | 1200 | 1200
menu | 8997 | 8997
delivery_partners | 2000 | 2000
payments | 20469 | 20469
customer_feedback | 14180 | 14180
promotions | 300 | 300
cities | 25 | 25
traffic | 18335 | 18335
weather | 18264 | 18264
ABT rows: 20475

2. MONEY TOTALS (orders table)
SUM FinalAmount: 8020380.1
SUM FoodCost: 7828881
SUM DeliveryFee: 561459
SUM Discount: 761403.95
SUM GST: 391444.05
ABT SUM FinalAmount: 8020380.1

3. ORDERS PER STATUS
OrderStatus
Cancelled              4569
Delivered             11390
Delivered Late         2154
Food Not Delivered     2362
Name: count, dtype: int64

4. DELIVERY TIME
AVG DeliveryTimeMinutes, original: 38.57
AVG DeliveryTimeMinutes, after cleaning: 36.76
Orders above 200 min in the original: 143

5. CUSTOMERS
Customers: 12000
Pincode = 0, original: 126
Pincode = 0, after cleaning: 0
Missing Pincode, original: 276
Missing Pincode, a

In [48]:
print("ABT shape:", abt_final.shape)
print("Duplicated OrderID:", abt_final["OrderID"].duplicated().sum())
print("Same row count as MySQL orders (20475):", len(abt_final) == 20475)
print("ABT SUM FinalAmount (MySQL: 8020380.10):", round(abt_final["FinalAmount"].sum(), 2))
print()

print("ABT orders per status:")
print(abt_final["OrderStatus"].value_counts().sort_index().to_string())
print()

print("Order date range:", abt_final["OrderDate"].min().date(), "to", abt_final["OrderDate"].max().date())
print()

num = abt_final.select_dtypes(include="number")
neg = (num < 0).sum()
print("Numeric columns with negative values:")
print(neg[neg > 0] if (neg > 0).any() else "none")
print()

missing = abt_final.isnull().sum()
print("Columns with missing values:", (missing > 0).sum(), "of", abt_final.shape[1])
print(missing[missing > 0].sort_values(ascending=False).to_string())

ABT shape: (20475, 68)
Duplicated OrderID: 0
Same row count as MySQL orders (20475): True
ABT SUM FinalAmount (MySQL: 8020380.10): 8020380.1

ABT orders per status:
OrderStatus
Cancelled              4569
Delivered             11390
Delivered Late         2154
Food Not Delivered     2362

Order date range: 2023-01-01 to 2024-12-31

Numeric columns with negative values:
none

Columns with missing values: 27 of 68
CouponCode               13268
PromoStartDate           13268
PromoEndDate             13268
DiscountPercentage       13268
CampaignName             13268
Rainfall                  7694
Temperature               7640
Humidity                  7572
AverageSpeed              7522
CustomerTenure            6749
PartnerExperienceDays     6730
CustomerRating            6700
DeliveryRating            6537
FoodRating                6410
Sentiment                 6410
CityAverageIncome         1634
RestaurantRating           685
PaymentStatus              405
PreferredCuisine          

In [49]:
import matplotlib.pyplot as plt
import seaborn as sns

print(abt.shape)
print(abt.columns.tolist())

(20475, 68)
['OrderID', 'CustomerID', 'RestaurantID', 'DeliveryPartnerID', 'OrderDate', 'OrderTime', 'DeliveryTimeMinutes', 'FoodCost', 'DeliveryFee', 'Discount', 'CouponCode', 'GST', 'FinalAmount', 'OrderStatus', 'PaymentMethod', 'CustomerAge', 'CustomerGender', 'CustomerCity', 'State', 'RegistrationDate', 'Membership', 'TotalOrders', 'PreferredCuisine', 'Cuisine', 'RestaurantCity', 'Area', 'RestaurantRating', 'AverageCost', 'RestaurantType', 'Latitude', 'Longitude', 'PartnerAge', 'VehicleType', 'PartnerJoiningDate', 'PartnerCity', 'PartnerRating', 'CompletedDeliveries', 'PartnerAvgDeliveryTime', 'ItemCount', 'TotalQuantity', 'ItemsTotalPrice', 'PaymentStatus', 'CustomerRating', 'DeliveryRating', 'FoodRating', 'Sentiment', 'Temperature', 'Rainfall', 'Humidity', 'AverageSpeed', 'Region', 'CityPopulation', 'CityAverageIncome', 'DiscountPercentage', 'CampaignName', 'PromoStartDate', 'PromoEndDate', 'OrderHour', 'PeakHour', 'DayOfWeek', 'WeekendOrder', 'OrderMonth', 'MealTime', 'CustomerT

In [50]:
print([c for c in abt.columns if 'raffic' in c or 'eather' in c or 'ain' in c])

['Rainfall']


In [51]:
print(abt.shape)
print(abt.head(3).T.tail(25))

(20475, 68)
                                         0                    1  \
DeliveryRating                         2.0                  3.0   
FoodRating                             1.0                  4.0   
Sentiment                         Negative             Positive   
Temperature                            NaN                 30.7   
Rainfall                               NaN               328.65   
Humidity                               NaN                 80.0   
AverageSpeed                          23.4                  NaN   
Region                               South                North   
CityPopulation                    13187628              3736453   
CityAverageIncome                1609572.0             745869.0   
DiscountPercentage                    50.0                 30.0   
CampaignName            Republic Day Offer       Payday Delight   
PromoStartDate         2022-08-17 00:00:00  2024-11-25 00:00:00   
PromoEndDate           2022-08-29 00:00:00  2024-1

In [52]:
print(abt.shape)
print(abt.head(3).T.tail(25))

(20475, 68)
                                         0                    1  \
DeliveryRating                         2.0                  3.0   
FoodRating                             1.0                  4.0   
Sentiment                         Negative             Positive   
Temperature                            NaN                 30.7   
Rainfall                               NaN               328.65   
Humidity                               NaN                 80.0   
AverageSpeed                          23.4                  NaN   
Region                               South                North   
CityPopulation                    13187628              3736453   
CityAverageIncome                1609572.0             745869.0   
DiscountPercentage                    50.0                 30.0   
CampaignName            Republic Day Offer       Payday Delight   
PromoStartDate         2022-08-17 00:00:00  2024-11-25 00:00:00   
PromoEndDate           2022-08-29 00:00:00  2024-1

In [53]:
print([c for c in abt.columns if 'raffic' in c or 'eather' in c or 'ain' in c])

['Rainfall']


In [54]:
import os
print(os.listdir('../data/cleaned'))
print(abt.columns.tolist())

['cities.csv', 'customers.csv', 'customer_feedback.csv', 'delivery_partners.csv', 'menu.csv', 'orders.csv', 'orders_weather_traffic.csv', 'order_items.csv', 'payments.csv', 'promotions.csv', 'restaurants.csv', 'traffic.csv', 'weather.csv']
['OrderID', 'CustomerID', 'RestaurantID', 'DeliveryPartnerID', 'OrderDate', 'OrderTime', 'DeliveryTimeMinutes', 'FoodCost', 'DeliveryFee', 'Discount', 'CouponCode', 'GST', 'FinalAmount', 'OrderStatus', 'PaymentMethod', 'CustomerAge', 'CustomerGender', 'CustomerCity', 'State', 'RegistrationDate', 'Membership', 'TotalOrders', 'PreferredCuisine', 'Cuisine', 'RestaurantCity', 'Area', 'RestaurantRating', 'AverageCost', 'RestaurantType', 'Latitude', 'Longitude', 'PartnerAge', 'VehicleType', 'PartnerJoiningDate', 'PartnerCity', 'PartnerRating', 'CompletedDeliveries', 'PartnerAvgDeliveryTime', 'ItemCount', 'TotalQuantity', 'ItemsTotalPrice', 'PaymentStatus', 'CustomerRating', 'DeliveryRating', 'FoodRating', 'Sentiment', 'Temperature', 'Rainfall', 'Humidity',

In [55]:
cols = ['OrderID', 'Rainfall', 'AverageSpeed', 'Temperature', 'Humidity']
abt[cols].to_csv('../data/cleaned/orders_weather_traffic.csv', index=False)
print(abt[cols].shape)
print(abt[cols].isna().sum())
print(abt[cols].describe())

(20475, 5)
OrderID            0
Rainfall        7694
AverageSpeed    7522
Temperature     7640
Humidity        7572
dtype: int64
             OrderID      Rainfall  AverageSpeed   Temperature      Humidity
count   20475.000000  12781.000000  12953.000000  12835.000000  12903.000000
mean   110252.059927     33.931503     23.190924     26.371853     59.166089
std      5918.525469     65.572781      9.955590      6.415255     12.982950
min    100001.000000      0.000000      3.000000      5.000000     10.000000
25%    105129.500000      1.050000     15.950000     21.750000     51.000000
50%    110252.000000      6.800000     23.000000     27.000000     59.000000
75%    115378.500000     32.400000     30.100000     31.100000     67.500000
max    120500.000000    871.900000     54.800000     47.400000    100.000000


In [56]:
export_cols = ['OrderID', 'Rainfall', 'AverageSpeed', 'Temperature', 'Humidity']
abt[export_cols].to_csv('../data/cleaned/orders_weather_traffic.csv', index=False)
print(abt[export_cols].shape)

(20475, 5)


In [57]:
check = pd.read_csv('../data/cleaned/orders_weather_traffic.csv')
print(check.shape, check['OrderID'].is_unique)

(20475, 5) True


In [58]:
export_cols = ['OrderID', 'Rainfall', 'AverageSpeed', 'Temperature', 'Humidity']
abt[export_cols].to_csv('../data/cleaned/orders_weather_traffic.csv', index=False)
print(abt[export_cols].shape)

(20475, 5)


In [59]:
check = pd.read_csv('../data/cleaned/orders_weather_traffic.csv')
print(check.shape, check['OrderID'].is_unique)

(20475, 5) True
